In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%%RecordEvent
# %load_ext cudf.pandas
# Import Visualization FrameWork Tools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import missingno as msno
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as mpatches
import time


In [ ]:
%%RecordEvent
%%time
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/new_notebooks/nyc-airbnb/AB_NYC_2019.csv")
df.head()

In [ ]:
%%RecordEvent
%%time
### cell 1 ###

df.shape

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

factor = 10
df = pd.concat([df]*factor)

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

# check null-value and each columns' dtype to do EDA.
print(df.info())

In [ ]:
%%RecordEvent
%%time
### cell 4 ###

df = df.drop(["id","last_review","name","host_name"],axis=1)
df.head()

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

df = df.drop(df[df['availability_365']==0].index,axis=0)
df = df.reset_index()
df.head()

## <div style="font-family:Serif;color:#DB3A49; padding:10px; font-size:35px">Fill Missing Value

In [ ]:
%%RecordEvent
%%time
### cell 6 ###

df["reviews_per_month"] = df['reviews_per_month'].fillna(0)
df.head()

In [ ]:
%%RecordEvent
%%time
### cell 7 ###

df.isnull().sum()

In [ ]:
%%RecordEvent
%%time
### cell 8 ###

df.shape

In [12]:
# def eda_map(df,feature, ax):
#     plt.figure(figsize=(15,15))
#     image = Image.open("/kaggle/input/new-york-city-airbnb-open-data/New_York_City_.png") # Load the image
#     df.plot(kind='scatter', x='longitude', y='latitude', label='AirBnB On service', c=str(feature), ax=ax, # Create the scatter plot
#             cmap=plt.get_cmap('jet'), colorbar=True, alpha=0.4, zorder=5, s=1)
#     ax.imshow(image,extent=[-74.258, -73.7, 40.49,40.92]) # Set the extent of the image

In [13]:
# plt.figure(figsize=(20,15))
# ax = plt.gca()
# eda_map(df,'price',ax)

In [ ]:
%%RecordEvent
%%time
### cell 9 ###

nyc_sub_1 = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']

x_1 = nyc_sub_1
y_1 = []
for i_1 in x_1:
    y_1.append(df[df['neighbourhood_group']==i_1]['price'].values.tolist())
y_1

In [ ]:
%%RecordEvent
%%time
### cell 10 ###

nyc_sub_2 = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
# --- End of sample data creation ---


# The optimized one-liner using groupby
# .apply(list) collects all values for each group into a list
grouped_prices = df.groupby('neighbourhood_group')['price'].apply(list)

# The result is a pandas Series, with boroughs as the index
# and the list of prices as the values.
# print(grouped_prices)
#
# Output would look like:
# neighbourhood_group
# Bronx              [175, 229, 403, 114, 308, 401, 381, 169, 14...
# Brooklyn           [187, 134, 421, 112, 179, 344, 185, 335, 41...
# Manhattan          [376, 269, 396, 429, 332, 287, 237, 275, 43...
# Queens             [487, 310, 342, 290, 488, 303, 442, 334, 16...
# Staten Island      [410, 423, 222, 172, 497, 180, 290, 391, 16...

# To get the exact same output as your original code (a list of lists in a specific order)
# you can reindex the result and convert it to a list.
y_optimized = grouped_prices.reindex(nyc_sub_2).tolist()
y_optimized

In [ ]:
%%RecordEvent
%%time
### cell 11 ###

df['price'].values[0]

In [ ]:
%%RecordEvent
%%time
### cell 12 ###

# Seperate by each Price
df["price_group"] = 0
for idx,pri in enumerate(df['price'][:100]):
    if df.loc[idx, 'price'] < 50:
        df.loc[idx, 'price_group'] = 0
    elif 50 <= df.loc[idx, 'price'] < 250:
        df.loc[idx, 'price_group'] = 1
    elif 250 <= df.loc[idx, 'price'] < 500:
        df.loc[idx, 'price_group'] = 2
    elif 500 <= df.loc[idx, 'price'] < 750:
        df.loc[idx, 'price_group'] = 3
    elif 750 <= df.loc[idx, 'price'] < 1000:
        df.loc[idx, 'price_group'] = 4
    else:
        df.loc[idx, 'price_group'] = 5

In [ ]:
%%RecordEvent
%%time
### cell 13 ###

suburb_area = [59100000,183400000,464900000,109000000,151500000]
distance_room_to_room = []
for i_2, city_1 in enumerate(nyc_sub_2):
    distance_room_to_room.append((suburb_area[i_2]/len(df[df['neighbourhood_group'] == city_1]))*(1/1000))

distance_room_to_room

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

total_amount_sub = []
total_amount_sub_50_250 = []
aver_price_total = []
aver_price_50_250 = []
for i_3, city_2 in enumerate(nyc_sub_2):
    req = df[(df['neighbourhood_group'] == city_2)&(df['price']>50)&(df['price']<250)]
    total_amount_sub.append(len(df[df['neighbourhood_group'] == city_2]))
    total_amount_sub_50_250.append(len(req))
    aver_price_total.append(round(np.mean(df[df['neighbourhood_group'] == city_2]['price']),3))
    aver_price_50_250.append(round(np.mean(req['price']),3))
total_amount_sub, total_amount_sub_50_250, aver_price_total, aver_price_50_250

In [ ]:
%%RecordEvent
%%time
### cell 15 ###

nei_x_1 = []
nei_y_1 = []
colors_1 = []

for nei_1 in df['neighbourhood'].value_counts().index:
    nei_x_1.append(nei_1)
    nei_y_1.append(np.mean(df[df['neighbourhood']==nei_1]['price'].values))

for neigh_1 in (nei_x_1):
    if df[df['neighbourhood']==neigh_1]['neighbourhood_group'].values[0] == 'Manhattan':
        colors_1.append("red")
    elif df[df['neighbourhood']==neigh_1]['neighbourhood_group'].values[0] == 'Brooklyn':
        colors_1.append("blue")
    elif df[df['neighbourhood']==neigh_1]['neighbourhood_group'].values[0] == 'Queens':
        colors_1.append("green")
    elif df[df['neighbourhood']==neigh_1]['neighbourhood_group'].values[0] == 'Bronx':
        colors_1.append("orange")
    elif df[df['neighbourhood']==neigh_1]['neighbourhood_group'].values[0] == 'Staten Island':
        colors_1.append("purple")

nei_x_1, nei_y_1, colors_1

In [ ]:
%%RecordEvent
%%time
### cell 16 ###

nei_x_aver_h_1 = []
nei_x_aver_l_1 = []
nei_y_aver_h_1 = []
nei_y_aver_l_1 = []
for nei_2 in df['neighbourhood'].value_counts().index:
    if np.mean(df[df['neighbourhood']==nei_2]['price'].values) >= round(np.mean(nei_y_1),3):
        nei_x_aver_h_1.append(nei_2)
        nei_y_aver_h_1.append(np.mean(df[df['neighbourhood']==nei_2]['price'].values))
    elif np.mean(df[df['neighbourhood']==nei_2]['price'].values) < round(np.mean(nei_y_1),3):
        nei_x_aver_l_1.append(nei_2)
        nei_y_aver_l_1.append(np.mean(df[df['neighbourhood']==nei_2]['price'].values))

nei_x_aver_h_1, nei_x_aver_l_1, nei_y_aver_h_1, nei_y_aver_l_1

In [ ]:
%%RecordEvent
%%time
### cell 17 ###

colors_2 = []
for neigh_2 in (nei_x_aver_h_1):
    if df[df['neighbourhood']==neigh_2]['neighbourhood_group'].values[0] == 'Manhattan':
        colors_2.append("red")
    elif df[df['neighbourhood']==neigh_2]['neighbourhood_group'].values[0] == 'Brooklyn':
        colors_2.append("blue")
    elif df[df['neighbourhood']==neigh_2]['neighbourhood_group'].values[0] == 'Queens':
        colors_2.append("green")
    elif df[df['neighbourhood']==neigh_2]['neighbourhood_group'].values[0] == 'Bronx':
        colors_2.append("orange")
    elif df[df['neighbourhood']==neigh_2]['neighbourhood_group'].values[0] == 'Staten Island':
        colors_2.append("purple")
colors_2

In [ ]:
%%RecordEvent
%%time
### cell 18 ###

colors_3 = []
for neigh_3 in (nei_x_aver_h_1):
    if df[df['neighbourhood']==neigh_3]['neighbourhood_group'].values[0] == 'Manhattan':
        colors_3.append("red")
    elif df[df['neighbourhood']==neigh_3]['neighbourhood_group'].values[0] == 'Brooklyn':
        colors_3.append("blue")
    elif df[df['neighbourhood']==neigh_3]['neighbourhood_group'].values[0] == 'Queens':
        colors_3.append("green")
    elif df[df['neighbourhood']==neigh_3]['neighbourhood_group'].values[0] == 'Bronx':
        colors_3.append("orange")
    elif df[df['neighbourhood']==neigh_3]['neighbourhood_group'].values[0] == 'Staten Island':
        colors_3.append("purple")
colors_3

In [ ]:
%%RecordEvent
%%time
### cell 19 ###

x_2 = df['calculated_host_listings_count'].value_counts().sort_index().index.astype(str)
y_2 = df['calculated_host_listings_count'].value_counts().sort_index().values

x_2, y_2